In [81]:
import pandas as pd
import numpy as np

In [82]:
from _counties import get_counties_df
counties_df = get_counties_df()
counties_df

,CO_FIPS,CO_NAME
0,5,CACHE
1,13,DUCHESNE
2,11,DAVIS
3,27,MILLARD
4,51,WASATCH
5,3,BOX ELDER
6,57,WEBER
7,23,JUAB
8,39,SANPETE
9,53,WASHINGTON


In [83]:
# read excel file into df
file_path = r"data\GPI-BaseYear-202503\GPI 2023 Statewide Estimates By County for Base Years (20250307).xlsx"

# Read **first** sheet (sheet_name=0)
base_year_2023_df = pd.read_excel(file_path, sheet_name=0, usecols=['COUNTY','YEAR','POP_ALL','POP_HH','POP_GQ','HH'])

base_year_2023_df['CO_NAME'] = base_year_2023_df['COUNTY'].str.upper()

base_year_2023_df.drop(columns=['COUNTY'], inplace=True)

base_year_2023_df = base_year_2023_df.merge(counties_df, on='CO_NAME')

display(base_year_2023_df)


,YEAR,POP_ALL,POP_HH,POP_GQ,HH,CO_NAME,CO_FIPS
0,2023,7.313582e+03,7.014582e+03,299,2280.837261,BEAVER,1
1,2023,6.125078e+04,6.108478e+04,166,20024.421000,BOX ELDER,3
2,2023,1.416997e+05,1.372327e+05,4467,44622.605274,CACHE,5
3,2023,2.065359e+04,2.053759e+04,116,7861.554559,CARBON,7
4,2023,9.977219e+02,9.977219e+02,0,405.249787,DAGGETT,9
5,2023,3.773785e+05,3.749495e+05,2429,120087.090579,DAVIS,11
6,2023,2.011192e+04,2.002492e+04,87,6560.578404,DUCHESNE,13
7,2023,1.003523e+04,9.977233e+03,58,3756.874670,EMERY,15
8,2023,5.140647e+03,4.936647e+03,204,1877.981383,GARFIELD,17
9,2023,9.842422e+03,9.686422e+03,156,3984.921939,GRAND,19


In [84]:
gpi_year_age_df = pd.read_csv("data/GPI-SE-Data-20251121/utah_proj_v2025_sya_2020_2065.csv")
gpi_2025_age_df = gpi_year_age_df[gpi_year_age_df['Year']==2025]
gpi_2025_age_df

,Geography,Year,Age,Pop
505,Beaver County,2025,0,94
506,Beaver County,2025,1,101
507,Beaver County,2025,2,105
508,Beaver County,2025,3,116
509,Beaver County,2025,4,97
...,...,...,...,...
135335,State of Utah,2025,96,895
135336,State of Utah,2025,97,653
135337,State of Utah,2025,98,434
135338,State of Utah,2025,99,291


In [85]:
# df columns expected:
# Geography | Year | Age | Pop

# 1. Define your age-group bins
bins = [0, 17, 64, 200]          # max age large enough to include all ages
labels = ["POP_00_17_pct", "POP_18_64_pct", "POP_65P_pct"]

# 2. Assign each record to an age group
gpi_2025_age_df["AgeGroup"] = pd.cut(gpi_2025_age_df["Age"].astype(int), bins=bins, labels=labels, right=True)

# 3. Aggregate
gpi_2025_age_groups_df = (
    gpi_2025_age_df.groupby(["Geography", "Year", "AgeGroup"], as_index=False, observed=False)
      .agg({"Pop": "sum"})
      .rename(columns={"AgeGroup": "Age"})
)

gpi_2025_age_groups_df = gpi_2025_age_groups_df[gpi_2025_age_groups_df['Geography'].str.contains(' County')]

gpi_2025_age_groups_df['CO_NAME'] = gpi_2025_age_groups_df['Geography'].str.replace(' County','').str.upper()

gpi_2025_age_groups_df.drop(columns=['Geography'], inplace=True)

display(gpi_2025_age_groups_df)


C:\Users\bhereth\AppData\Local\Temp\ipykernel_25096\1503458674.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gpi_2025_age_df["AgeGroup"] = pd.cut(gpi_2025_age_df["Age"].astype(int), bins=bins, labels=labels, right=True)


,Year,Age,Pop,CO_NAME
0,2025,POP_00_17_pct,1976,BEAVER
1,2025,POP_18_64_pct,4313,BEAVER
2,2025,POP_65P_pct,1153,BEAVER
3,2025,POP_00_17_pct,16901,BOX ELDER
4,2025,POP_18_64_pct,35793,BOX ELDER
...,...,...,...,...
85,2025,POP_18_64_pct,1331,WAYNE
86,2025,POP_65P_pct,634,WAYNE
87,2025,POP_00_17_pct,64562,WEBER
88,2025,POP_18_64_pct,168122,WEBER


In [86]:
# get percent of total in each Age Category
# 1. Compute total population by CO_NAME + Year
totals = (
    gpi_2025_age_groups_df
        .groupby(["CO_NAME", "Year"], as_index=False)
        .agg(TotalPop=("Pop", "sum"))
)

# 2. Merge totals back to the age-group DataFrame
gpi_2025_age_dist_df = gpi_2025_age_groups_df.merge(totals, on=["CO_NAME", "Year"])

# 3. Compute percent of total
gpi_2025_age_dist_df["PctOfTotal"] = gpi_2025_age_dist_df["Pop"] / gpi_2025_age_dist_df["TotalPop"]

gpi_2025_age_dist_df = (
    gpi_2025_age_dist_df[["CO_NAME", "Year", "Age", "PctOfTotal"]]
        .pivot(index=["CO_NAME"], columns="Age", values="PctOfTotal")
        .reset_index()
)

gpi_2025_age_dist_df

Age,CO_NAME,POP_00_17_pct,POP_18_64_pct,POP_65P_pct
0,BEAVER,0.265520,0.579549,0.154931
1,BOX ELDER,0.274469,0.581272,0.144258
2,CACHE,0.250284,0.642158,0.107558
3,CARBON,0.233831,0.566604,0.199565
4,DAGGETT,0.181078,0.533062,0.285860
5,DAVIS,0.268565,0.609046,0.122389
6,DUCHESNE,0.290552,0.567528,0.141920
7,EMERY,0.239841,0.565510,0.194648
8,GARFIELD,0.202801,0.534228,0.262971
9,GRAND,0.189774,0.576540,0.233686


# Apply 2025 split to 2023

In [90]:
pop_df = pd.merge(base_year_2023_df, gpi_2025_age_dist_df, on='CO_NAME')

pop_df['TOTPOP'] = np.round(pop_df['POP_ALL']).astype(int)
pop_df['HHPOP'] = np.round(pop_df['POP_HH']).astype(int)
pop_df['GQPOP'] = pop_df['TOTPOP'] - pop_df['HHPOP']

pop_df['POP0017'] = np.round(pop_df['POP_ALL'] * pop_df['POP_00_17_pct']).astype(int)
pop_df['POP1864'] = np.round(pop_df['POP_ALL'] * pop_df['POP_18_64_pct']).astype(int)
pop_df['POP65P'] = pop_df['TOTPOP'] - pop_df['POP0017'] - pop_df['POP1864']

pop_df = pop_df[['CO_FIPS','YEAR','TOTPOP','GQPOP','HHPOP','POP0017','POP1864','POP65P','HH','CO_NAME']]

pop_melt_df = pop_df.melt(id_vars=['CO_FIPS','YEAR','CO_NAME'], var_name='TDM_VARIABLE', value_name='VALUE')
pop_melt_df['VALUE'] = pop_melt_df['VALUE'].astype(int)
pop_melt_df

,CO_FIPS,YEAR,CO_NAME,TDM_VARIABLE,VALUE
0,1,2023,BEAVER,TOTPOP,7314
1,3,2023,BOX ELDER,TOTPOP,61251
2,5,2023,CACHE,TOTPOP,141700
3,7,2023,CARBON,TOTPOP,20654
4,9,2023,DAGGETT,TOTPOP,998
...,...,...,...,...,...
198,49,2023,UTAH,HH,211379
199,51,2023,WASATCH,HH,12556
200,53,2023,WASHINGTON,HH,70844
201,55,2023,WAYNE,HH,947


In [91]:
pop_melt_df[['CO_FIPS','YEAR','TDM_VARIABLE','VALUE','CO_NAME']].to_csv('intermediate/tdm_population_by_county_2023.csv', index=False)